**Series 4 · ML Fundamentals** ‣ [04-1 ML คืออะไร](./04-1-ml-introduction.ipynb) ・ [04-2 Train/Test Split & Overfitting](./04-2-train-test-split-and-overfitting.ipynb) ・ [04-3 Pipeline & Data Leakage](./04-3-pipeline-and-data-leakage.ipynb)

---

# 🔗 04-3: Data Leakage และ Pipeline
### รวมทุกขั้นตอนของ Series 2 ให้เป็นโค้ดชุดเดียวที่ปลอดภัย

### 🎯 วัตถุประสงค์การเรียนรู้
1. รู้จัก Data Leakage 2 แบบ และตรวจจับได้
2. ใช้ `Pipeline` และ `ColumnTransformer` รวม Imputation + Encoding + Scaling + Model
3. ได้ **Template โค้ด ML** ที่นำไปใช้ทำข้อสอบได้ทันที

## 1. Data Leakage คืออะไร?

**Data Leakage** = โมเดลได้เห็นข้อมูลที่ **ไม่ควรรู้** ในตอนทำนายจริง ทำให้คะแนนตอนทดสอบสูงเกินจริง แต่พอใช้งานจริงกลับแย่

| ประเภท | ความหมาย | ตัวอย่าง |
|---|---|---|
| **Target Leakage** | Feature มีข้อมูลของคำตอบแฝงอยู่ | ใช้ `TotalPlayer_binned` ทำนาย `TotalPlayer` |
| **Train-Test Contamination** | ข้อมูล Test รั่วเข้าไปตอน Preprocess | `fit` Scaler/Imputer กับข้อมูลทั้งหมดก่อนแบ่ง |

> 🚩 **สัญญาณเตือน:** คะแนนดีเกินจริง (เช่น R² หรือ Accuracy เกิน 0.95 ในปัญหาที่ควรยาก) → ตรวจหา Leakage ก่อนดีใจ

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 📌 ถ้าใช้ Google Colab: อัปโหลดไฟล์ CSV ที่แผง Files (ซ้ายมือ)
#    แล้วเปลี่ยน DATA_PATH เป็น "golf_dataset_weather_tropical_1year.csv"
DATA_PATH = "Data/golf_dataset_weather_tropical_1year.csv"
df = pd.read_csv(DATA_PATH)
print("ขนาดข้อมูล:", df.shape)

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression

num_cols = ['SunshineDuration', 'Temperature', 'Humidity', 'WindDirection', 'WindSpeed', 'Pressure']
cat_cols = ['Outlook']
y = df['TotalPlayer']
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ขนาดข้อมูล: (365, 15)


## 2. ทดลอง Target Leakage
`TotalPlayer_binned` ถูกสร้างโดยแบ่งช่วง `TotalPlayer` (บท 02-4) จึงบอกคำตอบโดยตรง

In [2]:
print(df.groupby('TotalPlayer_binned')['TotalPlayer'].agg(['min', 'max']).sort_values('min'))

                    min  max
TotalPlayer_binned          
Very Few             14   33
Few                  35   49
Moderate             50   64
Many                 65   79
Lots                 80   94
Extreme              95   99


In [3]:
X_ok   = df[num_cols]
X_leak = pd.get_dummies(df[num_cols + ['TotalPlayer_binned']], columns=['TotalPlayer_binned'])

r2_ok   = cross_val_score(LinearRegression(), X_ok,   y, cv=kf, scoring='r2').mean()
r2_leak = cross_val_score(LinearRegression(), X_leak, y, cv=kf, scoring='r2').mean()
print(f"R² ปกติ          : {r2_ok:.3f}")
print(f"R² มี Leakage    : {r2_leak:.3f}   ← สูงเกินจริง! ตอนใช้งานจริงเราไม่มีคอลัมน์นี้")

R² ปกติ          : 0.543
R² มี Leakage    : 0.960   ← สูงเกินจริง! ตอนใช้งานจริงเราไม่มีคอลัมน์นี้


## 3. Train-Test Contamination

ลำดับที่ **ถูกต้อง** คือ **แบ่งข้อมูลก่อน แล้วค่อย `fit` ตัว Preprocess กับ Train เท่านั้น**

```
❌ ผิด:   Scale ทั้งหมด → แบ่ง Train/Test      (Test รั่วเข้าไปในค่า mean/std)
✅ ถูก:   แบ่ง Train/Test → fit Scaler กับ Train → transform ทั้ง Train และ Test
```

ถ้าทำเองทีละขั้น โค้ดจะยาวและพลาดง่าย (โดยเฉพาะตอนทำ Cross-Validation ที่ต้องทำซ้ำทุก Fold)
→ ทางแก้คือ **Pipeline**

## 4. Pipeline: ต่อทุกขั้นตอนเป็นสายพานเดียว

```
ข้อมูลดิบ → [Imputer] → [Scaler] → [Model] → คำทำนาย
```
เมื่อเรียก `pipe.fit(X_train, y_train)` ทุกขั้นจะ `fit` กับ Train เท่านั้นโดยอัตโนมัติ
และเมื่อใช้กับ `cross_val_score` ก็จะ `fit` ใหม่ในทุก Fold ให้เอง → **ไม่มี Leakage**

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

X = df[num_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # บท 02-2
    ('scaler',  StandardScaler()),                    # บท 02-5
    ('model',   KNeighborsRegressor(n_neighbors=10)),
])

pipe.fit(X_train, y_train)
print("R² Test (KNN + Scaling):", round(pipe.score(X_test, y_test), 3))

no_scale = KNeighborsRegressor(n_neighbors=10).fit(X_train, y_train)
print("R² Test (KNN ไม่ Scale) :", round(no_scale.score(X_test, y_test), 3))

R² Test (KNN + Scaling): 0.571
R² Test (KNN ไม่ Scale) : 0.354


> 🔍 KNN ใช้ระยะทาง ถ้าไม่ Scale ตัวแปรที่ค่าใหญ่ (`WindDirection` 0–360, `Pressure` ~1000) จะครอบงำระยะทาง
> นี่คือเหตุผลเชิงปฏิบัติที่เราเรียน Scaling ในบท 02-5

## 5. ColumnTransformer: จัดการตัวเลขและหมวดหมู่แยกกัน

ข้อมูลจริงมีทั้งตัวเลขและข้อความ ซึ่งต้อง Preprocess ต่างกัน

| ชนิดคอลัมน์ | ขั้นตอน | อ้างอิง |
|---|---|---|
| ตัวเลข (Numerical) | เติมค่าหาย (median) → Scale | บท 02-2, 02-5 |
| หมวดหมู่ (Categorical) | เติมค่าหาย (most_frequent) → One-Hot Encode | บท 02-2, 02-3 |

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore')),   # เจอหมวดใหม่ใน Test ก็ไม่ error
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe,     num_cols),
    ('cat', categorical_pipe, cat_cols),
])

# ดูผลลัพธ์หลัง Preprocess (เพื่อความเข้าใจเท่านั้น ปกติไม่ต้องดู)
X_full = df[num_cols + cat_cols]
preview = preprocessor.fit_transform(X_full)
pd.DataFrame(preview, columns=preprocessor.get_feature_names_out()).head().round(2)

,num__SunshineDuration,num__Temperature,num__Humidity,num__WindDirection,num__WindSpeed,num__Pressure,cat__Outlook_overcast,cat__Outlook_rainy,cat__Outlook_sunny
0,-0.76,0.04,0.63,-1.07,-0.51,0.58,0.0,1.0,0.0
1,2.10,2.06,-0.35,1.45,-0.92,1.74,0.0,0.0,1.0
2,2.47,1.37,0.19,1.41,-0.21,-1.34,0.0,0.0,1.0
3,-1.02,-0.09,-0.57,1.05,-0.17,0.30,1.0,0.0,0.0
4,0.20,1.21,-1.56,-1.61,1.14,0.98,0.0,0.0,1.0


## 6. Pipeline เต็มรูปแบบ: Preprocess + Model

In [6]:
X = df[num_cols + cat_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

full_pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', LinearRegression()),
])
full_pipe.fit(X_train, y_train)

cv = cross_val_score(full_pipe, X_train, y_train, cv=kf, scoring='r2')
print(f"CV R² (Train)   : {cv.mean():.3f} ± {cv.std():.3f}")
print(f"R² Test         : {full_pipe.score(X_test, y_test):.3f}")
print(f"(เทียบบท 04-2 ที่ไม่มี Outlook: Test R² ≈ 0.60)")

CV R² (Train)   : 0.796 ± 0.070


R² Test         : 0.857
(เทียบบท 04-2 ที่ไม่มี Outlook: Test R² ≈ 0.60)


> ✅ เพิ่ม `Outlook` (ซึ่งต้อง Encode) ทำให้ R² Test เพิ่มจาก ~0.60 เป็น ~0.86 เพราะวันฝนตกคนมาน้อยมาก และ Pipeline จัดการ Encode ให้ในบรรทัดเดียว

## 7. เปลี่ยนโมเดลง่ายๆ: เปรียบเทียบหลายโมเดลพร้อมกัน
เพราะ Preprocess ถูกแยกไว้ใน `preprocessor` แล้ว เราสลับแค่ขั้น `model` ได้เลย (ข้อสอบมักให้เปรียบเทียบ)

In [7]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0),
    'KNN (k=10)':        KNeighborsRegressor(n_neighbors=10),
    'Decision Tree':     DecisionTreeRegressor(max_depth=4, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42),
}

rows = []
for name, m in models.items():
    p = Pipeline([('prep', preprocessor), ('model', m)])
    cv = cross_val_score(p, X_train, y_train, cv=kf, scoring='r2')
    p.fit(X_train, y_train)
    rows.append({'Model': name, 'CV R² mean': cv.mean(), 'CV R² std': cv.std(),
                 'Test R²': p.score(X_test, y_test)})

pd.DataFrame(rows).sort_values('CV R² mean', ascending=False).round(3).reset_index(drop=True)

,Model,CV R² mean,CV R² std,Test R²
0,Linear Regression,0.796,0.070,0.857
1,Ridge,0.796,0.072,0.857
2,Random Forest,0.777,0.076,0.837
3,KNN (k=10),0.753,0.066,0.781
4,Decision Tree,0.749,0.089,0.823


> 💡 **เลือกโมเดลจาก CV R² mean** (บน Train) แล้วรายงาน Test R² เป็นผลสุดท้าย
> ไม่ใช่เลือกจาก Test R² (ทบทวนบท 04-2 ข้อ 5)

## 8. ทำนายข้อมูลใหม่ด้วย Pipeline
ส่งข้อมูลดิบเข้าไปได้เลย Pipeline จะ Impute + Encode + Scale ให้เอง (แม้จะมีค่าหาย)

In [8]:
new_days = pd.DataFrame({
    'SunshineDuration': [10.0, 1.0, np.nan],
    'Temperature':      [30.0, 24.0, 27.0],
    'Humidity':         [65,   90,   75],
    'WindDirection':    [180.0, 180.0, 90.0],
    'WindSpeed':        [2.0,  2.0,  3.0],
    'Pressure':         [1009.0, 1009.0, 1008.0],
    'Outlook':          ['sunny', 'rainy', 'overcast'],
})
new_days['Predicted_Players'] = full_pipe.predict(new_days).round(0)
new_days

,SunshineDuration,Temperature,Humidity,WindDirection,WindSpeed,Pressure,Outlook,Predicted_Players
0,10.0,30.0,65,180.0,2.0,1009.0,sunny,72.0
1,1.0,24.0,90,180.0,2.0,1009.0,rainy,23.0
2,NaN,27.0,75,90.0,3.0,1008.0,overcast,60.0


## 9. 🧰 Template ML สำหรับข้อสอบ (คัดลอกไปใช้ได้เลย)

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1) โหลดข้อมูล
df = pd.read_csv("data.csv")

# 2) กำหนด Target และ Feature (ตัดคอลัมน์ ID / Leakage ออก)
target = "ชื่อคอลัมน์คำตอบ"
X = df.drop(columns=[target, "คอลัมน์_ID"])
y = df[target]
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 3) แบ่งข้อมูล (Classification ให้เพิ่ม stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4) Preprocessor
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

# 5) Pipeline + Model
pipe = Pipeline([("prep", preprocessor), ("model", ใส่โมเดลที่นี่())])

# 6) Cross-Validation บน Train
cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2")   # หรือ "accuracy", "f1"
print(f"CV: {cv.mean():.3f} ± {cv.std():.3f}")

# 7) เทรนจริง + วัดผลบน Test ครั้งเดียว
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
```
> Metrics อย่างละเอียด (MAE, RMSE, Confusion Matrix, F1, ROC-AUC) จะต่อท้ายข้อ 7 ใน Series 5–7

---

## 10. สรุป

| สิ่งที่ต้องจำ | เหตุผล |
|---|---|
| ตัดคอลัมน์ที่ **สร้างจาก Target** หรือ **รู้ได้หลังเกิดเหตุ** | กัน Target Leakage |
| **แบ่งข้อมูลก่อน** Preprocess เสมอ | กัน Train-Test Contamination |
| ใช้ **Pipeline + ColumnTransformer** | Preprocess ถูกต้องอัตโนมัติ, CV ไม่ Leak, สลับโมเดลง่าย |
| `OneHotEncoder(handle_unknown='ignore')` | Test มีหมวดใหม่ก็ไม่ error |
| คะแนนสูงผิดปกติ | ตรวจ Leakage ก่อนเสมอ |

## 📝 แบบฝึกหัด
ใช้ Template ข้างบนทำ **Classification** ทำนาย `IsBusy` (`TotalPlayer >= 60`)
โดยใช้ `num_cols + cat_cols` เดิม, `stratify=y`, `StratifiedKFold` และ `scoring='accuracy'`
เปรียบเทียบ `LogisticRegression`, `KNeighborsClassifier`, `RandomForestClassifier`

In [9]:
# เฉลย
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

X = df[num_cols + cat_cols]
y_cls = (df['TotalPlayer'] >= 60).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN (k=10)':          KNeighborsClassifier(n_neighbors=10),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
}
rows = []
for name, m in clf_models.items():
    p = Pipeline([('prep', preprocessor), ('model', m)])
    cv = cross_val_score(p, Xc_train, yc_train, cv=skf, scoring='accuracy')
    p.fit(Xc_train, yc_train)
    rows.append({'Model': name, 'CV Acc mean': cv.mean(), 'CV Acc std': cv.std(),
                 'Test Acc': p.score(Xc_test, yc_test)})
pd.DataFrame(rows).sort_values('CV Acc mean', ascending=False).round(3).reset_index(drop=True)

,Model,CV Acc mean,CV Acc std,Test Acc
0,KNN (k=10),0.791,0.039,0.808
1,Random Forest,0.791,0.039,0.795
2,Logistic Regression,0.791,0.043,0.795


---
⬅️ **บทก่อนหน้า:** [04-2 Train/Test Split & Overfitting](./04-2-train-test-split-and-overfitting.ipynb) ・ ➡️ **Series ถัดไป:** 05 Supervised – Regression